# Customer Churn — a logistic-regression baseline

Before reaching for a heavier framework, it's worth having a simple, transparent
baseline for churn prediction — something we fully understand and can sanity-check
by hand. This notebook builds one end to end: simulate a small labelled dataset,
standardize the features, fit logistic regression with gradient descent, and
evaluate on a held-out split.

Everything here uses only the Python standard library, so the model is completely
explicit — no hidden defaults from a library — which makes it a good reference
point to compare a "real" model against later.

## Setup

A couple of imports and a fixed random seed so the run is reproducible. `sigmoid`
is the logistic link we'll reuse throughout.

In [2]:
import math
import random
import statistics

random.seed(7)

def sigmoid(z):
    return 1.0 / (1.0 + math.exp(-z))

print("Environment ready")

Environment ready


## Configuration

One place for dataset size, hold-out fraction, and gradient-descent settings.

In [3]:
N_SAMPLES = 400
TEST_FRACTION = 0.25
LEARNING_RATE = 0.05
EPOCHS = 200

## Data

We don't have a production export on hand, so we simulate a churn dataset that's
realistic enough to reason about. Each customer has two numeric drivers — think
tenure and monthly charges — and the churn label is drawn from a hidden linear
relationship plus noise, which is roughly what logistic regression assumes.

We then standardize each feature to zero mean and unit variance so the two
drivers are on the same scale and gradient descent converges evenly.

In [ ]:
def make_sample():
    x1 = random.gauss(0, 1)   # tenure
    x2 = random.gauss(0, 1)   # monthly charges
    logit = 1.4 * x1 - 1.1 * x2 + random.gauss(0, 0.5)
    y = 1 if logit > 0 else 0
    return [x1, x2], y

data = [make_sample() for _ in range(N_SAMPLES)]
print(f"Generated {len(data)} samples; churn rate = {statistics.mean([y for _, y in data]):.2f}")

Generated 400 samples; churn rate = 0.49


In [ ]:
col = lambda i: [row[i] for row, _ in data]
m1, s1 = statistics.mean(col(0)), statistics.pstdev(col(0)) or 1.0
m2, s2 = statistics.mean(col(1)), statistics.pstdev(col(1)) or 1.0

def scale(row):
    return [(row[0] - m1) / s1, (row[1] - m2) / s2]

scaled = [(scale(row), y) for row, y in data]
print(f"Standardized {len(scaled[0][0])} features")

Standardized 2 features


## Model

Logistic regression is just a linear score passed through the sigmoid. We start
the weights at zero — one per feature — plus a bias term.

In [6]:
weights = [0.0 for _ in range(len(scaled[0][0]))]
bias = 0.0

def score(features):
    return sum(w * f for w, f in zip(weights, features)) + bias

## Training

Batch gradient descent over the training split. On each pass we compute the
prediction error for every sample and step the weights down the gradient of the
log-loss. It's slow compared to a vectorized library, but it's easy to follow.

In [7]:
train_n = int(len(scaled) * (1 - TEST_FRACTION))
train, test = scaled[:train_n], scaled[train_n:]

for _ in range(EPOCHS):
    for features, y in train:
        error = sigmoid(score(features)) - y
        for i in range(len(weights)):
            weights[i] -= LEARNING_RATE * error * features[i]
        bias -= LEARNING_RATE * error

print(f"Trained on {len(train)} samples for {EPOCHS} epochs")

Trained on 300 samples for 200 epochs


## Evaluation

The real question is how well the model separates churners it hasn't seen. We
report accuracy on both splits and inspect the learned weights — their signs
should line up with the hidden rule the data was generated from.

In [8]:
def confusion_counts(rows):
    tp = fp = tn = fn = 0
    for features, y in rows:
        pred = sigmoid(score(features)) > 0.5
        actual = bool(y)
        if pred and actual:
            tp += 1
        elif pred and not actual:
            fp += 1
        elif not pred and actual:
            fn += 1
        else:
            tn += 1
    return tp, fp, tn, fn

def precision_f1(rows):
    tp, fp, tn, fn = confusion_counts(rows)
    precision = tp / (tp + fp) if (tp + fp) else 0.0
    recall = tp / (tp + fn) if (tp + fn) else 0.0
    f1 = 2 * precision * recall / (precision + recall) if (precision + recall) else 0.0
    return precision, f1

for name, rows in (("Train", train), ("Test", test)):
    precision, f1 = precision_f1(rows)
    print(f"{name} precision: {precision:.3f} | F1: {f1:.3f}")

tp, fp, tn, fn = confusion_counts(test)
print("\nTest confusion matrix:")
print(f"{'':>12}{'pred 1':>8}{'pred 0':>8}")
print(f"{'actual 1':>12}{tp:>8}{fn:>8}")
print(f"{'actual 0':>12}{fp:>8}{tn:>8}")
print("Weights:", [round(w, 3) for w in weights], "| bias:", round(bias, 3))


Train precision: 0.910 | F1: 0.907
Test precision: 0.857 | F1: 0.878

Test confusion matrix:
              pred 1  pred 0
    actual 1      36       4
    actual 0       6      54
Weights: [4.883, -4.398] | bias: 0.068


## Persisting the model

Save the learned parameters so the model can be reloaded for scoring later
without retraining.

In [9]:
with open("model_weights.txt", "w") as f:
    f.write(repr({"weights": weights, "bias": bias}))
print("Saved model_weights.txt")

Saved model_weights.txt


## Notes & next steps

The baseline lands around 90% accuracy on this synthetic data, and the weight
signs recover the generating rule — a reassuring sanity check.

From here the usual improvements apply: add L2 regularization, try interaction
features, log the training loss to confirm convergence, calibrate the decision
threshold rather than assuming 0.5, and validate with a proper cross-validation
loop before trusting the numbers.